In [30]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

from loguru import logger


class BinaryClassifierWrapper:
    """Wrapper for consistent prediction interface"""

    def __init__(
        self,
        model,
        model_type,
        feature_names,
        label_encoder=None,
        feature_encoders=None
    ):
        self.model = model
        self.model_type = model_type
        self.feature_names = feature_names
        self.label_encoder = label_encoder
        self.feature_encoders = feature_encoders or {}

    def predict(
        self,
        model_input: pd.DataFrame,
        return_probs: bool = False,
        return_both: bool = False
    ):
        df = model_input.copy()

        # Encode categorical columns if needed
        for col, encoder in self.feature_encoders.items():
            if col in df.columns:
                df[col] = encoder.transform(df[col].astype(str))

        X = df[self.feature_names]

        probs = self.model.predict_proba(X)[:, 1]

        binary_preds = (probs >= 0.5).astype(int)

        if self.label_encoder is not None:
            final_preds = self.label_encoder.inverse_transform(binary_preds)
        else:
            final_preds = binary_preds

        max_probs = np.maximum(probs, 1 - probs)

        if return_both:
            return pd.DataFrame(
                {
                    "probability": max_probs,
                    "prediction": final_preds,
                }
            )

        if return_probs:
            return max_probs

        return final_preds


class GenericBinaryClassifierTrainer:
    """Generic trainer for binary classification models"""

    SUPPORTED_MODELS = {
        "random_forest": RandomForestClassifier,
        "decision_tree": DecisionTreeClassifier,
        "logistic_regression": LogisticRegression,
    }

    def __init__(
        self,
        config: dict,
        model_type: str,
    ):
        if model_type not in self.SUPPORTED_MODELS:
            raise ValueError(
                f"model_type must be one of "
                f"{list(self.SUPPORTED_MODELS.keys())}, "
                f"got '{model_type}'"
            )

        self.config = config
        self.model_type = model_type
        self.model = None
        self.feature_names = None

    def prepare_data(
        self,
        data: pd.DataFrame,
        target_col: str,
        feature_cols: list[str] | None = None,
        test_size: float = 0.2,
        random_state: int = 42,
    ):
        logger.info(
            f"Preparing training data for {self.model_type}..."
        )

        if feature_cols is None:
            feature_cols = [
                col for col in data.columns
                if col != target_col
            ]

        self.feature_names = feature_cols

        X = data[feature_cols]
        y = data[target_col]

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=test_size,
            random_state=random_state,
            stratify=y,
        )

        logger.info(
            f"Training set: {X_train.shape}, "
            f"Test set: {X_test.shape}"
        )

        logger.info(
            f"Class distribution - Train: "
            f"{y_train.value_counts().to_dict()}"
        )

        logger.info(
            f"Class distribution - Test: "
            f"{y_test.value_counts().to_dict()}"
        )

        return X_train, X_test, y_train, y_test

    def train(
        self,
        X_train: pd.DataFrame,
        y_train: pd.Series,
        X_test: pd.DataFrame,
        y_test: pd.Series,
        params: dict | None = None,
    ):
        """
        Train model

        Returns:
            trained sklearn model
        """

        logger.info(
            f"Starting {self.model_type} model training..."
        )

        if params is None:
            params = self.config.get(
                self.model_type,
                {}
            )

        if self.feature_names is None:
            raise ValueError(
                "Please call prepare_data() first."
            )

        model_class = self.SUPPORTED_MODELS[
            self.model_type
        ]

        self.model = model_class(**params)

        self.model.fit(
            X_train,
            y_train
        )

        train_score = self.model.score(
            X_train,
            y_train
        )

        test_score = self.model.score(
            X_test,
            y_test
        )

        logger.info("Training complete.")
        logger.info(
            f"Train accuracy: {train_score:.4f}"
        )
        logger.info(
            f"Test accuracy: {test_score:.4f}"
        )

        self._log_feature_importance()

        return self.model

    def _log_feature_importance(self):
        """Show feature importance"""

        if self.model is None:
            return

        importance_dict = {}

        if self.model_type in [
            "random_forest",
            "decision_tree"
        ]:
            if hasattr(
                self.model,
                "feature_importances_"
            ):
                importance_dict = dict(
                    zip(
                        self.feature_names,
                        self.model.feature_importances_
                    )
                )

        elif self.model_type == "logistic_regression":
            if hasattr(
                self.model,
                "coef_"
            ):
                importance_dict = dict(
                    zip(
                        self.feature_names,
                        np.abs(
                            self.model.coef_[0]
                        )
                    )
                )

        if importance_dict:

            logger.info(
                "Top Feature Importances:"
            )

            sorted_importance = sorted(
                importance_dict.items(),
                key=lambda x: x[1],
                reverse=True
            )

            for feature, score in sorted_importance[:10]:
                logger.info(
                    f"{feature}: {score:.6f}"
                )

    def save_model(
        self,
        model_path: str,
        label_encoder=None,
        feature_encoders=None,
    ):
        """
        Save trained model
        """

        if self.model is None:
            raise ValueError(
                "Model has not been trained."
            )

        wrapper = BinaryClassifierWrapper(
            model=self.model,
            model_type=self.model_type,
            feature_names=self.feature_names,
            label_encoder=label_encoder,
            feature_encoders=feature_encoders,
        )

        save_object = {
            "wrapper": wrapper,
            "model": self.model,
            "model_type": self.model_type,
            "feature_names": self.feature_names,
            "label_encoder": label_encoder,
            "feature_encoders": feature_encoders,
        }

        model_path = Path(model_path)

        model_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        joblib.dump(
            save_object,
            model_path
        )

        logger.info(
            f"Model saved to {model_path}"
        )

    @staticmethod
    def load_model(model_path: str):
        """
        Load saved model
        """

        model_data = joblib.load(
            model_path
        )

        logger.info(
            f"Model loaded from {model_path}"
        )

        return model_data

In [31]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

CSV_PATH = "/home/schaffen/Workspace/Project/Lead_classification/data/data/processed/df_processed.csv"

df = pd.read_csv(CSV_PATH)

X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=50,
    max_depth=5,
    random_state=42
)

model.fit(X_train, y_train)

print("Train:", model.score(X_train, y_train))
print("Test:", model.score(X_test, y_test))

joblib.dump(model, "model.pkl")

Train: 1.0
Test: 1.0


['model.pkl']

In [32]:
loaded = GenericBinaryClassifierTrainer.load_model(
    "/home/schaffen/Workspace/Project/Lead_classification/model/notebooks/model.pkl"
)

preds = loaded.predict(X_test)
print(preds[:10])

2026-06-03 14:40:03.473 | INFO     | __main__:load_model:320 - Model loaded from /home/schaffen/Workspace/Project/Lead_classification/model/notebooks/model.pkl


[2 1 2 1 2 1 2 2 2 3]


In [33]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Predict
y_pred = loaded.predict(X_test)

# Metrics
acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted")
recall = recall_score(y_test, y_pred, average="weighted")
f1 = f1_score(y_test, y_pred, average="weighted")

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1-score : 1.0000

Classification Report:
              precision    recall  f1-score   support

           1       1.00      1.00      1.00        67
           2       1.00      1.00      1.00        88
           3       1.00      1.00      1.00        45

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200


Confusion Matrix:
[[67  0  0]
 [ 0 88  0]
 [ 0  0 45]]
